# SPATIAL INTELLIGENCE - GEOMETRY IMPORT & SETUP

Import the prison floor plan from OBJ, clean the geometry, create a grid overlay, slice into a shell, and derive the analysis graph.

## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy version

In [ ]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Configuration

In [ ]:
import os

renderer = "vscode"

# Paths
BASE_DIR = r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Assignment-02_RamonGarcia"
OBJ_PATH = os.path.join(BASE_DIR, "Prison_floorplan.obj")
BREP_PATH = os.path.join(BASE_DIR, "prison_clean.brep")
ASSETS_DIR = os.path.join(BASE_DIR, "Assets")
os.makedirs(ASSETS_DIR, exist_ok=True)

GRID_SIZE = 2

# Image saving - the orchestrator sets this to True
SAVE_IMAGES = True

def save_fig(fig, filename):
    # Save a plotly figure to the assets directory as PNG.
    if not SAVE_IMAGES or fig is None:
        return
    try:
        path = os.path.join(ASSETS_DIR, filename)
        fig.write_image(path, width=1600, height=1200, scale=2)
        print(f"Saved: {path}")
    except Exception as e:
        print(f"Could not save {filename}: {e}")

## 4. Import the prison floor plan from OBJ

`Topology.ByOBJPath` returns a list of clusters. The OBJ mesh gets triangulated on import, so we merge all triangles into a shell, then extract the external and internal boundaries to reconstruct the floor plan as a single clean face.

In [ ]:
result = Topology.ByOBJPath(OBJ_PATH)
print(f"OBJ returned {len(result)} items")

# Collect all triangulated faces from the clusters
all_faces = []
for i, item in enumerate(result):
    if Topology.IsInstance(item, "Cluster"):
        cf = Cluster.Faces(item)
        if cf:
            all_faces.extend(cf)
    elif Topology.IsInstance(item, "Face"):
        all_faces.append(item)

print(f"Found {len(all_faces)} triangulated faces")

# Merge triangles into a shell
mesh_shell = Shell.ByFaces(all_faces)
print(f"Shell created from mesh triangles")

# Extract external boundary (outer perimeter) and internal boundaries (courtyards/voids)
eb = Shell.ExternalBoundary(mesh_shell)
ib_list = Shell.InternalBoundaries(mesh_shell)
print(f"External boundary: {len(Topology.Vertices(eb))} vertices")
print(f"Internal boundaries (holes): {len(ib_list) if ib_list else 0}")

# Reconstruct as a single face with holes
if ib_list:
    floor_plan = Face.ByWires(eb, ib_list)
else:
    floor_plan = Face.ByWire(eb)

floor_plan = Topology.RemoveCollinearEdges(floor_plan)
print(f"\nFloor plan: {len(Topology.Vertices(floor_plan))} vertices")

## 5. Show the floor plan

In [ ]:
fig = Topology.Show(floor_plan,
              camera=[0, 0, 6],
              faceColor=[210, 210, 250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)
save_fig(fig, "01_floor_plan.png")

## 6. Create a grid overlay

In [ ]:
b_r = Wire.BoundingRectangle(floor_plan)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
print(f"Bounding box: {round(width, 1)} x {round(length, 1)} units")

uRange = list(range(0, int(width) + GRID_SIZE, GRID_SIZE))
vRange = list(range(0, int(length) + GRID_SIZE, GRID_SIZE))
grid = Grid.EdgesByDistances(floor_plan, clip=True, uRange=uRange, vRange=vRange)

## 7. Show the floor plan with grid overlay

In [ ]:
fig = Topology.Show(floor_plan, grid,
              camera=[0, 0, 6],
              faceColor=[210, 210, 250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)
save_fig(fig, "02_grid_overlay.png")

## 8. Slice the floor plan with the grid to create a shell

In [ ]:
shell = Topology.Slice(floor_plan, grid)
faces = Topology.Faces(shell)
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_" + str(i + 1))
    f = Topology.SetDictionary(f, d)
print(f"Shell created with {len(faces)} faces")

## 9. Show the resulting shell

In [ ]:
fig = Topology.Show(shell,
              camera=[0, 0, 6],
              faceColor=[210, 210, 250],
              faceOpacity=0.9,
              edgeColor="black",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)
save_fig(fig, "03_sliced_shell.png")

## 10. Derive the analysis graph from the shell

In [ ]:
# Derive the analysis graph from the shell
analysis_graph = Graph.ByTopology(shell)
g_verts = Graph.Vertices(analysis_graph)
print(f"Analysis graph: {len(g_verts)} vertices, {len(Graph.Edges(analysis_graph))} edges")

## 11. Show the analysis graph

In [ ]:
fig = Topology.Show(analysis_graph,
              camera=[0, 0, 6],
              vertexSize=4,
              vertexColor="red",
              edgeColor="lightgrey",
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)
save_fig(fig, "04_analysis_graph.png")

## 12. Save the cleaned geometry for subsequent notebooks

In [ ]:
status = Topology.ExportToBREP(floor_plan, path=BREP_PATH, overwrite=True)
print(f"BREP export: {status}")
print(f"Saved to: {BREP_PATH}")